# Dados brutos da pesquisa Focus em 2026-10-23

Este notebook imprime **todos os registros cuja chave `Data` é `2026-10-23`** nos 12 recursos estatísticos do BCB presentes no snapshot local. Cada linha exibida abaixo é o **objeto JSON original** extraído da resposta comprimida preservada em `dados/bruto/`: números, `null`, grafia e ordem dos campos são mantidos. O envelope OData (`@odata.context` e `value`) fica nos arquivos `.json.gz`.

A tabela inicial mostra a proveniência e a contagem de cada recurso. `DatasReferencia`, o 13º recurso, não tem campo `Data` e por isso não entra neste recorte. O notebook não acessa a API nem altera a base. As saídas são longas porque nenhum registro é omitido.


**Limite dos dados:** “brutos” aqui significa a resposta JSON original da API de estatísticas, sem transformação pelo projeto. Cada objeto contém medidas como `Media`, `Mediana`, `Minimo`, `Maximo`, `DesvioPadrao` e, quando disponível, `numeroRespondentes`; **não é uma previsão individual de um respondente**. O BCB lista o recurso `ExpectativasMercadoInstituicoes` como desativado por risco de quebra de confidencialidade da autoria dos microdados ([catálogo oficial](https://dadosabertos.bcb.gov.br/dataset/expectativas-mercado/resource/2231aecd-3dbe-4e1a-bae6-674d3be225a1)).


## Preparação e conferência dos arquivos

O SQLite é usado apenas para descobrir de qual coleta veio a data em cada recurso. Os valores impressos são lidos diretamente dos JSONs originais. Para cada página, o código confere o SHA-256 do manifesto e compara as contagens com o JSON analisado.


In [ ]:
projeto <- if (file.exists("dados/atual.json")) "." else if (file.exists("../dados/atual.json")) ".." else stop("Abra na raiz do projeto ou na pasta notebooks/")
if (requireNamespace("renv", quietly = TRUE)) renv::load(project = normalizePath(projeto), quiet = TRUE)
suppressPackageStartupMessages({library(DBI); library(RSQLite); library(jsonlite); library(digest)})
data_alvo <- "2026-10-23"
atual <- fromJSON(file.path(projeto, "dados/atual.json"))
inventario <- fromJSON(file.path(projeto, atual$manifestos, "inventario.json"), simplifyVector = FALSE)
recursos <- setdiff(names(inventario), "DatasReferencia")
con <- dbConnect(SQLite(), file.path(projeto, atual$banco))
cat("Data:", data_alvo, "| snapshot:", atual$coleta, "| recursos:", length(recursos), "\n")


In [ ]:
carregar_recurso <- function(recurso) {
  tabela <- as.character(dbQuoteIdentifier(con, recurso))
  origens <- dbGetQuery(con, sprintf("SELECT DISTINCT coleta FROM %s WHERE Data = ?", tabela),
                        params = list(data_alvo))$coleta
  if (length(origens) != 1L) stop("Origem ausente ou ambígua: ", recurso)
  coleta <- origens[[1]]
  pasta <- file.path(projeto, "dados/manifestos", coleta, recurso)
  manifestos <- list.files(pasta, pattern = "\\.json$", full.names = TRUE)
  if (!length(manifestos)) stop("Manifesto ausente: ", recurso)
  paginas <- do.call(rbind, lapply(manifestos, function(arquivo) {
    m <- fromJSON(arquivo, simplifyVector = FALSE)
    selecionadas <- Filter(function(pg) !is.null(pg$data_min) && !is.null(pg$data_max) &&
                             pg$data_min <= data_alvo && pg$data_max >= data_alvo, m$paginas)
    if (!length(selecionadas)) return(NULL)
    do.call(rbind, lapply(selecionadas, function(pg)
      data.frame(janela = m$janela$id, arquivo = pg$arquivo,
                 sha256 = pg$sha256, url = pg$url)))
  }))
  if (is.null(paginas) || !nrow(paginas)) stop("Nenhuma página para ", recurso)
  linhas <- unlist(lapply(seq_len(nrow(paginas)), function(i) {
    caminho <- file.path(projeto, "dados/bruto", coleta, recurso,
                         paginas$janela[[i]], paginas$arquivo[[i]])
    if (!file.exists(caminho)) stop("Arquivo bruto ausente: ", caminho)
    if (!identical(digest(file = caminho, algo = "sha256"), paginas$sha256[[i]]))
      stop("SHA-256 não confere: ", caminho)
    comprimido <- readBin(caminho, "raw", n = file.info(caminho)$size)
    texto <- rawToChar(memDecompress(comprimido, type = "gzip"))
    resposta <- fromJSON(texto, simplifyVector = FALSE)
    # Os 12 esquemas estatísticos têm objetos planos. Esta extração conserva
    # exatamente o texto de cada objeto, inclusive zeros finais e null.
    objetos <- regmatches(texto, gregexpr("\\{[^{}]*\\}", texto, perl = TRUE))[[1]]
    if (length(objetos) != length(resposta$value))
      stop("Estrutura JSON inesperada em ", caminho)
    selecionados <- objetos[grepl(paste0('"Data":"', data_alvo, '"'), objetos, fixed = TRUE)]
    esperados <- sum(vapply(resposta$value, function(linha)
      identical(linha$Data, data_alvo), logical(1)))
    if (length(selecionados) != esperados)
      stop("Filtro da data não confere em ", caminho)
    selecionados
  }), use.names = FALSE)
  list(coleta = coleta, paginas = paginas, linhas = linhas)
}
brutos <- setNames(lapply(recursos, carregar_recurso), recursos)
dbDisconnect(con)
resumo <- data.frame(
  recurso = recursos,
  coleta = vapply(brutos, function(x) x$coleta, character(1)),
  paginas = vapply(brutos, function(x) nrow(x$paginas), integer(1)),
  registros = vapply(brutos, function(x) length(x$linhas), integer(1)))
stopifnot(all(resumo$registros > 0L))
print(resumo, row.names = FALSE)
cat("Total:", sum(resumo$registros), "objetos JSON originais\\n")


## Registros originais por recurso

Cada saída contém **todos** os objetos da data, um JSON por linha. Para achar o arquivo de origem, consulte `brutos[["NomeDoRecurso"]]$paginas`; a coluna `url` traz a URL da requisição original.


In [ ]:
mostrar_brutos <- function(recurso) {
  x <- brutos[[recurso]]
  cat("Recurso:", recurso, "| Data:", data_alvo, "| registros:", length(x$linhas), "\n")
  cat(paste(x$linhas, collapse = "\n"), "\n", sep = "")
  invisible(NULL)
}


## Mensais — `ExpectativaMercadoMensais`


In [ ]:
mostrar_brutos("ExpectativaMercadoMensais")


## Selic — `ExpectativasMercadoSelic`


In [ ]:
mostrar_brutos("ExpectativasMercadoSelic")


## Trimestrais — `ExpectativasMercadoTrimestrais`


In [ ]:
mostrar_brutos("ExpectativasMercadoTrimestrais")


## Anuais — `ExpectativasMercadoAnuais`


In [ ]:
mostrar_brutos("ExpectativasMercadoAnuais")


## Inflação 12 meses — `ExpectativasMercadoInflacao12Meses`


In [ ]:
mostrar_brutos("ExpectativasMercadoInflacao12Meses")


## Inflação 13 a 24 meses — `ExpectativasMercadoInflacao24Meses`


In [ ]:
mostrar_brutos("ExpectativasMercadoInflacao24Meses")


## Top 5 mensais — `ExpectativasMercadoTop5Mensais`


In [ ]:
mostrar_brutos("ExpectativasMercadoTop5Mensais")


## Top 5 Selic — `ExpectativasMercadoTop5Selic`


In [ ]:
mostrar_brutos("ExpectativasMercadoTop5Selic")


## Top 5 trimestral — `ExpectativaMercadoTop5Trimestral`


In [ ]:
mostrar_brutos("ExpectativaMercadoTop5Trimestral")


## Top 5 anuais — `ExpectativasMercadoTop5Anuais`


In [ ]:
mostrar_brutos("ExpectativasMercadoTop5Anuais")


## Top 5 inflação 12 meses — `ExpectativasMercadoTop5Inflacao12Meses`


In [ ]:
mostrar_brutos("ExpectativasMercadoTop5Inflacao12Meses")


## Top 5 inflação 13 a 24 meses — `ExpectativasMercadoTop5Inflacao24Meses`


In [ ]:
mostrar_brutos("ExpectativasMercadoTop5Inflacao24Meses")


## Arquivos de origem

O objeto `brutos` conserva o caminho lógico de cada página no manifesto. O código abaixo imprime a lista de páginas e hashes usados nesta execução. Para abrir a resposta OData inteira, descomprima o `.json.gz` indicado, sem aplicar o filtro por `Data`.


In [ ]:
for (recurso in names(brutos)) {
  x <- brutos[[recurso]]
  cat("\n", recurso, " (coleta ", x$coleta, ")\n", sep = "")
  print(x$paginas[, c("janela", "arquivo", "sha256")], row.names = FALSE)
}
